# Overview

This notebook covers two assignment tasks:
1. **Demonstrate overfitting and underfitting**: based on the same CNN model, demonstrate overfitting and underfitting of the same model with almost the same training parameters.
- Underfitting (Model1) is demonstrated by training model for fewer epochs AND on the larger training subset (existing checkpoint is used).
- Overfitting (Model2) is demonstrated by training model for more epochs AND on the smaller training subset.

2. **Error analysis** consists of 2 parts:
- Error analysis of both models: confusion matrix analysis and visual inspection of misclassified examples, use of Top-N metrics.
- Comparing of the two models by investigating fixed and regressed cases.

In [ ]:
import copy
import pprint
import time
import math
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import wandb

import importlib
import src.cnn.cnn_utils as cnn_utils
import src.cnn.cnn_registry as cnn_registry
import src.cnn.cnn_models as cnn_models
import src.cnn.configs as configs
import src.cnn.cnn_paths as cnn_paths
from cnn.cnn_utils import compare_prediction_tables, top_n_fixed_cases

importlib.reload(cnn_utils)

# Shallow CNN

## Train Config

In [ ]:
# see configs.py
cfg1 = configs.ExperimentConfig()

# =========================================================
# DATASET CONFIG
# =========================================================
cfg1.dataset.dataset_dir = cnn_paths.DATASET_DIR  # see cnn_paths.py
cfg1.dataset.train_subdir = "train"
cfg1.dataset.val_subdir = "validate"
cfg1.dataset.test_subdir = None  # or "test" if you have it
cfg1.dataset.image_size = 224

# leave transforms as None -> default Resize + ToTensor pipeline
cfg1.dataset.train_transform = None
cfg1.dataset.eval_transform = None

# optional normalization
cfg1.dataset.normalize_mean = None
cfg1.dataset.normalize_std = None


# =========================================================
# MODEL CONFIG
# IMPORTANT: name is saved in MODEL_REGISTRY -> cnn_registry.py
# use @register_model("depth_cnn") decorator to add new models to the registry and make them available by name in the config
# =========================================================
cfg1.model.name = "shallow_model"
cfg1.model.kwargs = {
    "in_channels": 3,
    "num_classes": 10,
    "units": 128,
    "drop": 0.5,
}

# =========================================================
# TRAIN CONFIG
# =========================================================
cfg1.train.epochs = 30
cfg1.train.device = str(cnn_utils.get_device("auto"))
cfg1 = cnn_utils.configure_runtime_defaults(cfg1)
cfg1.train.non_blocking = True
cfg1.train.use_amp = False

# RECOMMENDED FOR CUDA
# cfg.train.use_amp = True

cfg1.train.grad_clip_norm = None
cfg1.train.best_metric = "val/accuracy"
cfg1.train.best_mode = "max"
cfg1.train.seed = 13

# =========================================================
# DATALOADER CONFIG
# =========================================================
cfg1.loader.batch_size = 16
cfg1.loader.num_workers = 0
cfg1.loader.pin_memory = (torch.device(cfg1.train.device).type == "cuda")

# RECOMMENDED FOR CUDA
# cfg.loader.pin_memory = True

cfg1.loader.train_shuffle = True
cfg1.loader.eval_shuffle = False
cfg1.loader.drop_last_train = False
cfg1.loader.drop_last_eval = False


# =========================================================
# LOSS CONFIG
# =========================================================
cfg1.loss.cls = nn.CrossEntropyLoss
cfg1.loss.kwargs = {}


# =========================================================
# OPTIMIZER CONFIG
# =========================================================
cfg1.optimizer.cls = torch.optim.Adam
cfg1.optimizer.kwargs = {
    "lr": 1e-3,
    "weight_decay": 1e-4,
}


# =========================================================
# SCHEDULER CONFIG
# Example: Reduce LR when validation loss plateaus
# =========================================================
cfg1.scheduler.cls = torch.optim.lr_scheduler.ReduceLROnPlateau
cfg1.scheduler.kwargs = {
    "mode": "min",
    "factor": 0.5,
    "patience": 2,
}
cfg1.scheduler.step_metric = "val/loss"

# =========================================================
# W&B CONFIG
# =========================================================
cfg1.wandb.enabled = True
cfg1.wandb.project = "MPW-CNN"
cfg1.wandb.entity = "MSE_DeLearn_SPR26"
cfg1.wandb.mode = "online"  # "online", "offline", or "disabled" for no logging
cfg1.wandb.log_confusion_matrix = True

# NOTE: use meaningful names for runs, so the difference is clear
cfg1.wandb.run_name = "OUfitting_" + time.strftime("%Y%m%d-%H%M%S")

# NOTE: use meaningful grouping, for example, by model or by task.
# E.g. task "Data augmentation" => experiments on light-mid-heavy augmentation
cfg1.wandb.group = "OUfitting"
cfg1.wandb.job_type = "train"

# NOTE: use meaningful tags to filter runs in UI
cfg1.wandb.tags = ["cnn", "fitting"]
cfg1.wandb.notes = ""

cfg1.wandb.log_epoch_metrics = True
cfg1.wandb.log_every_n_epochs = 1

cfg1.wandb.metric_allowlist = {
    "train/loss",
    "train/accuracy",
    "val/loss",
    "val/accuracy",
    "gap/accuracy",
    "gap/loss",
    "lr",
}

cfg1.wandb.summary_allowlist = {
    "best_epoch",
    "best_metric_name",
    "best_metric_value",
    "train_final/loss",
    "train_final/accuracy",
    "val_final/loss",
    "val_final/accuracy",
}

cfg1.wandb.watch_model = False
cfg1.wandb.watch_log = "all"
cfg1.wandb.watch_log_freq = 100

## Eval config

In [ ]:
cfg_eval1 = copy.deepcopy(cfg1)
cfg_eval1.train.device = "cpu"

# Model1

## Model1: Data and loaders

In [ ]:
datasets_dict = cnn_utils.load_datasets(cfg1.dataset)

train_dataset = datasets_dict["train"]
val_dataset = datasets_dict["val"]
train_loader = cnn_utils.make_train_loader(train_dataset, cfg1)
val_loader = cnn_utils.make_eval_loader(val_dataset, cfg1)

## Model1: Load pretrained from file

In [ ]:
checkpoint_file = "gradcam_20260408-185704_20260408_192332.pth"
ckpt_path = cnn_paths.CKPT_DIR  # in data/checkpoints/

model1, checkpoint = cnn_utils.load_model_for_inference(
    path=ckpt_path,
    filename=checkpoint_file,)

## Model1: Eval

In [ ]:
cfg_eval1 = copy.deepcopy(cfg1)
cfg_eval1.train.device = "cpu"

model1_cpu = model1.to("cpu").eval()

if torch.backends.mps.is_available():
    torch.mps.synchronize()
    torch.mps.empty_cache()

loader_for_cm = val_loader
split_name = "val"
class_names = loader_for_cm.dataset.classes

y_true1, y_pred1 = cnn_utils.predict_loader(model1_cpu, loader_for_cm, cfg_eval1)

print("structure of predictions/true labels:")
print("true label counts:", Counter(y_true1))
print("pred label counts:", Counter(y_pred1))
print("class_names:", class_names)

print("Unique y_true:", sorted(set(y_true1)))
print("Unique y_pred:", sorted(set(y_pred1)))

In [ ]:
cnn_utils.evaluate_model_on_loader(loader=val_loader, model=model1, cfg=cfg_eval1)

## Model1: Confusion matrix analysis

In [ ]:
fig_counts1 = cnn_utils.make_confusion_matrix_fig(
    y_true=y_true1,
    y_pred=y_pred1,
    class_names=class_names,
    normalize=None,
    title=f"{split_name.capitalize()} Confusion Matrix (counts)",
)

fig_norm1 = cnn_utils.make_confusion_matrix_fig(
    y_true=y_true1,
    y_pred=y_pred1,
    class_names=class_names,
    normalize="true",
    title=f"{split_name.capitalize()} Confusion Matrix (normalized)",
)

## Model1: Top-N metrics

In [ ]:
y_true_arr1 = np.asarray(y_true1)
y_pred_arr1 = np.asarray(y_pred1)

# # true=dog=3, pred=cat=0
# idxs1 = np.where((y_true_arr1 == 3) & (y_pred_arr1 == 0))[0]
# print(idxs1)

### Model1: Collect prediction records

In [ ]:
records_df1 = cnn_utils.collect_prediction_records(
    model=model1,
    dataset=val_dataset,
    cfg=cfg_eval1,
    class_names=class_names,
    batch_size=64,
)

### Model1: Top-10 most confident mistakes

In [ ]:
confident_mistakes1 = cnn_utils.top_n_most_confident_mistakes(records_df1, n=10)
cnn_utils.summarize_cases(confident_mistakes1)
cnn_utils.plot_case_grid(
    val_dataset,
    confident_mistakes1,
    title="Top-10 most confident mistakes, Model1",
)

### Model1: Top-10 least confident correct predictions

In [ ]:
fragile_correct1 = cnn_utils.top_n_least_confident_correct(records_df1, n=10)
cnn_utils.summarize_cases(fragile_correct1)
cnn_utils.plot_case_grid(
    val_dataset,
    fragile_correct1,
    title="Top-10 least confident correct predictions, Model1",
)

### Model1: Top-10 most uncertain overall

In [ ]:
uncertain_entropy1 = cnn_utils.top_n_most_uncertain_overall(records_df1, n=10, by="entropy")
cnn_utils.summarize_cases(uncertain_entropy1)
cnn_utils.plot_case_grid(
    val_dataset,
    uncertain_entropy1,
    title="Top-10 most uncertain_entropy samples overall, Model1",
)

### Model1: Most common confusion pairs

In [ ]:
pairs_df1 = cnn_utils.most_common_confusion_pairs(records_df1, top_k=10)
pairs_df1

# Model2

## Model2: Config
Same config as for Model1, but with more epochs.

In [ ]:
cfg2 = copy.deepcopy(cfg1)
cfg2.train.epochs = 80
cfg2.loader.batch_size = 4

cfg_eval2 = copy.deepcopy(cfg2)
cfg_eval2.train.device = "cpu"

## Model2: Data and loaders
To overfit the model, we will train it on **fraction of all train images** used for Model1.

In [ ]:
datasets_dict = cnn_utils.load_datasets(cfg2.dataset)

train_dataset = datasets_dict["train"]
val_dataset = datasets_dict["val"]

train_loader2, train_dataset2 = cnn_utils.make_stratified_fraction_train_loader(
    train_dataset,
    cfg2,
    fraction=0.005,   # 0.005 = 12 imgs/class
    seed=56,
    verbose=True,
)

val_loader = cnn_utils.make_eval_loader(val_dataset, cfg2)

## Model2: Train

In [ ]:
# Model builder from config

model2 = cnn_registry.build_model(cfg2.model)

print(model2)
print(f"Trainable parameters: {cnn_utils.get_num_parameters(model2):,}")

Use new train loader with fraction of data used before.

In [ ]:
model2, history2, result2 = cnn_utils.train_and_evaluate_model(
    model=model2,
    train_loader=train_loader2,
    val_loader=val_loader,
    cfg=cfg2,
    run_name=cfg2.wandb.run_name,
)

## Model2: Eval

### On VAL loader

In [ ]:
model2_cpu = model2.to("cpu").eval()

if torch.backends.mps.is_available():
    torch.mps.synchronize()
    torch.mps.empty_cache()

loader_for_cm = val_loader
split_name = "val"
class_names = loader_for_cm.dataset.classes

y_true2, y_pred2 = cnn_utils.predict_loader(model2_cpu, loader_for_cm, cfg_eval2)

print("structure of predictions/true labels:")
print("true label counts:", Counter(y_true2))
print("pred label counts:", Counter(y_pred2))
print("class_names:", class_names)

print("Unique y_true:", sorted(set(y_true2)))
print("Unique y_pred:", sorted(set(y_pred2)))

In [ ]:
pprint.pprint(result2)

### On TRAIN loader

In [ ]:
y_true2_train, y_pred2_train = cnn_utils.predict_loader(model2_cpu, train_loader2, cfg_eval2)

print("structure of predictions/true labels:")
print("true label counts:", Counter(y_true2_train))
print("pred label counts:", Counter(y_pred2_train))
print("class_names:", class_names)

print("Unique y_true:", sorted(set(y_true2_train)))
print("Unique y_pred:", sorted(set(y_pred2_train)))

## Model2: Save model to file

In [ ]:
ckpt_path = cnn_paths.CKPT_DIR  # in data/checkpoints/

# Checkpoint file with timestamp of when the training was completed (in train_eval)
checkpoint_file = str(cfg2.wandb.run_name) + '_' + result2["timestamp"] + ".pth"

cnn_utils.save_checkpoint(
    path=ckpt_path,
    filename=checkpoint_file,
    model=model2,
    cfg=cfg2,
    extra={"result": result2},
)

### Log checkpoint as artifact in W&B

In [ ]:
if cfg2.wandb.enabled and cfg2.wandb.mode != "disabled":
    with wandb.init(
        project=cfg2.wandb.project,
        entity=cfg2.wandb.entity,
        job_type="log_checkpoint",
        mode=cfg2.wandb.mode,
        id=result2["wandb_run_id"], # add to the same run!
        resume="must",              # for that, run is resumed
    ):
        cnn_utils.log_checkpoint_artifact(
            path=ckpt_path,
            filename=checkpoint_file,
            cfg=cfg2,
            result=result2,
        )

## Model2: Confusion matrix analysis

In [ ]:
fig_counts2 = cnn_utils.make_confusion_matrix_fig(
    y_true=y_true2,
    y_pred=y_pred2,
    class_names=class_names,
    normalize=None,
    title=f"{split_name.capitalize()} Confusion Matrix (counts)",
)

fig_norm2 = cnn_utils.make_confusion_matrix_fig(
    y_true=y_true2,
    y_pred=y_pred2,
    class_names=class_names,
    normalize="true",
    title=f"{split_name.capitalize()} Confusion Matrix (normalized)",
)

### Log confusion matrices to W&B

In [ ]:
with wandb.init(
    project=cfg2.wandb.project,
    entity=cfg2.wandb.entity,
    mode=cfg2.wandb.mode,
    id=result2["wandb_run_id"], # add to the same run!
    resume="must",              # for that, run is resumed
    job_type="log_confusion_matrix",
):
    try:
        cnn_utils.log_confusion_matrix(
            y_true=y_true2,
            y_pred=y_pred2,
            class_names=class_names,
            key=f"{split_name}/confusion_matrix_chart",
            title=f"{split_name.capitalize()} Confusion Matrix",
            split_table=False,
        )
    except Exception as e:
        wandb.log({
            f"{split_name}/confusion_matrix_chart_error": str(e)
        })
        print("W&B custom confusion matrix failed:", e)

    wandb.log({
        f"{split_name}/confusion_matrix_counts": wandb.Image(fig_counts2),
        f"{split_name}/confusion_matrix_normalized": wandb.Image(fig_norm2),
    })

    wandb.finish()

## Model2: Top-N metrics

In [ ]:
y_true_arr1 = np.asarray(y_true1)
y_pred_arr1 = np.asarray(y_pred1)

### Model2: Collect prediction records

In [ ]:
records_df2 = cnn_utils.collect_prediction_records(
    model=model2,
    dataset=val_dataset,
    cfg=cfg_eval2,
    class_names=class_names,
    batch_size=64,
)

### Model2: Top-10 most confident mistakes

In [ ]:
confident_mistakes2 = cnn_utils.top_n_most_confident_mistakes(records_df2, n=10)
cnn_utils.summarize_cases(confident_mistakes2)
cnn_utils.plot_case_grid(
    val_dataset,
    confident_mistakes2,
    title="Top-10 most confident mistakes, Model2",
)

### Model2: Top-10 least confident correct predictions

In [ ]:
fragile_correct2 = cnn_utils.top_n_least_confident_correct(records_df2, n=10)
cnn_utils.summarize_cases(fragile_correct2)
cnn_utils.plot_case_grid(
    val_dataset,
    fragile_correct2,
    title="Top-10 least confident correct predictions, Model2",
)

### Model2: Top-10 most uncertain overall

In [ ]:
uncertain_entropy2 = cnn_utils.top_n_most_uncertain_overall(records_df2, n=10, by="entropy")
cnn_utils.summarize_cases(uncertain_entropy2)
cnn_utils.plot_case_grid(
    val_dataset,
    uncertain_entropy2,
    title="Top-10 most uncertain_entropy samples overall, Model2",
)

### Model2: Most common confusion pairs

In [ ]:
pairs_df2 = cnn_utils.most_common_confusion_pairs(records_df2, top_k=10)
pairs_df2

# Compare Models

In [ ]:
comparison_df = cnn_utils.compare_prediction_tables(
    before_df=records_df2, # overfitted model
    after_df=records_df1, # underfitted model
)

## Top-N fixed and regressed cases

In [ ]:
fixed_cases = cnn_utils.top_n_fixed_cases(comparison_df, n=10)
cnn_utils.summarize_comparison_cases(fixed_cases)

In [ ]:
cnn_utils.plot_case_grid(
    val_dataset,
    cnn_utils.comparison_rows_to_case_grid_df(fixed_cases, use_after=True),
    title="Fixed cases (Model2 => Model1)",
)